In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv("processed_data/tickets_2M_final.csv")
print(df.shape)
df.head()

C:\Users\ricky\AppData\Local\Temp\ipykernel_20948\2910606393.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("processed_data/tickets_2M_final.csv")


(2000000, 13)


,ticket_id,subject,description,category,issue_type,priority,resolution,is_noisy,has_screenshot,has_logs,log_text,screenshot_path,input_text
0,TCKT_0000001,Cannot authenticate to DB on postgres,The app cannot login to the database using con...,Database,Authentication to DB Failed Variant 398,High,Verify DB credentials and access permissions.,False,False,False,NaN,NaN,Cannot authenticate to DB on postgres The app ...
1,TCKT_0000002,Cannot resolve host on gateway,I am getting host resolution errors while brow...,Network,DNS Resolution Error Variant 270,Medium,Flush DNS cache and check network settings.,False,False,False,NaN,NaN,Cannot resolve host on gateway I am getting ho...
2,TCKT_0000003,pass prompt appears again and again in outlook on,my syncing usage. not affecting outlook becaus...,Email,Outlook Password Prompt Loop Variant 107,High,Clear saved Outlook credentials and sign in ag...,True,False,True,Outlook auth popup loop detected,NaN,pass prompt appears again and again in outlook...
3,TCKT_0000004,Laptop not connecting to WiFi on lan,My laptop cannot connect to office WiFi and ke...,Network,WiFi Not Connecting Variant 405,Medium,"Restart WiFi adapter, forget network, and reco...",False,False,False,NaN,NaN,Laptop not connecting to WiFi on lan My laptop...
4,TCKT_0000005,on wrkng laptop,is failing. affecting is type this i screen be...,Hardware,Keyboard Not Working Variant 508,Medium,Reconnect keyboard driver or restart the device.,True,False,False,NaN,NaN,on wrkng laptop is failing. affecting is type ...


In [3]:
required_cols = [
    "ticket_id", "input_text", "category", "issue_type",
    "priority", "resolution", "has_screenshot", "screenshot_path"
]

df = df[required_cols].copy()

df["input_text"] = df["input_text"].fillna("").astype(str)
df["category"] = df["category"].fillna("").astype(str)
df["issue_type"] = df["issue_type"].fillna("").astype(str)
df["resolution"] = df["resolution"].fillna("").astype(str)

print(df.shape)
print(df.isnull().sum())
df.head()

(2000000, 8)
ticket_id                0
input_text               0
category                 0
issue_type               0
priority                 0
resolution               0
has_screenshot           0
screenshot_path    1980035
dtype: int64


,ticket_id,input_text,category,issue_type,priority,resolution,has_screenshot,screenshot_path
0,TCKT_0000001,Cannot authenticate to DB on postgres The app ...,Database,Authentication to DB Failed Variant 398,High,Verify DB credentials and access permissions.,False,NaN
1,TCKT_0000002,Cannot resolve host on gateway I am getting ho...,Network,DNS Resolution Error Variant 270,Medium,Flush DNS cache and check network settings.,False,NaN
2,TCKT_0000003,pass prompt appears again and again in outlook...,Email,Outlook Password Prompt Loop Variant 107,High,Clear saved Outlook credentials and sign in ag...,False,NaN
3,TCKT_0000004,Laptop not connecting to WiFi on lan My laptop...,Network,WiFi Not Connecting Variant 405,Medium,"Restart WiFi adapter, forget network, and reco...",False,NaN
4,TCKT_0000005,on wrkng laptop is failing. affecting is type ...,Hardware,Keyboard Not Working Variant 508,Medium,Reconnect keyboard driver or restart the device.,False,NaN


In [4]:
from sklearn.preprocessing import LabelEncoder

In [5]:
category_encoder = LabelEncoder()
issue_encoder = LabelEncoder()

df["category_label"] = category_encoder.fit_transform(df["category"])
df["issue_label"] = issue_encoder.fit_transform(df["issue_type"])

print("Unique categories:", len(category_encoder.classes_))
print("Unique issue types:", len(issue_encoder.classes_))

print(df[["category", "category_label", "issue_type", "issue_label"]].head())

Unique categories: 8
Unique issue types: 5000
   category  category_label                                issue_type  \
0  Database               1   Authentication to DB Failed Variant 398   
1   Network               4          DNS Resolution Error Variant 270   
2     Email               2  Outlook Password Prompt Loop Variant 107   
3   Network               4           WiFi Not Connecting Variant 405   
4  Hardware               3          Keyboard Not Working Variant 508   

   issue_label  
0          750  
1         1129  
2         3344  
3         4921  
4         2462  


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:
X = df["input_text"]
y_category = df["category_label"]
y_issue = df["issue_label"]

In [8]:
X_train, X_test, y_cat_train, y_cat_test, y_issue_train, y_issue_test = train_test_split(
    X, y_category, y_issue,
    test_size=0.2,
    random_state=42,
    stratify=y_category
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1600000
Test size: 400000


In [9]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (1600000, 9483)
X_test_tfidf shape: (400000, 9483)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [11]:
category_model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

category_model.fit(X_train_tfidf, y_cat_train)

LogisticRegression(max_iter=1000, n_jobs=-1)

In [12]:
cat_pred = category_model.predict(X_test_tfidf)

cat_acc = accuracy_score(y_cat_test, cat_pred)
print("Category Accuracy:", round(cat_acc, 4))
print()
print(classification_report(
    y_cat_test,
    cat_pred,
    target_names=category_encoder.classes_
))

Category Accuracy: 1.0

              precision    recall  f1-score   support

      Access       1.00      1.00      1.00     50756
    Database       1.00      1.00      1.00     50065
       Email       1.00      1.00      1.00     52152
    Hardware       1.00      1.00      1.00     49148
     Network       1.00      1.00      1.00     48410
     Printer       1.00      1.00      1.00     52716
    Security       1.00      1.00      1.00     48139
    Software       1.00      1.00      1.00     48614

    accuracy                           1.00    400000
   macro avg       1.00      1.00      1.00    400000
weighted avg       1.00      1.00      1.00    400000



In [13]:
from sklearn.svm import LinearSVC

In [ ]:
issue_model = LinearSVC()
issue_model.fit(X_train_tfidf, y_issue_train)

In [ ]:
issue_pred = issue_model.predict(X_test_tfidf)

issue_acc = accuracy_score(y_issue_test, issue_pred)
print("Issue Type Accuracy:", round(issue_acc, 4))